In [92]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [93]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,1:], df.iloc[:, 0])

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

label = LabelEncoder()
y_train = label.fit_transform(y_train)
y_test = label.transform(y_test)

print(y_test)


[0 0 1 0 1 0 0 0 0 1 0 0 0 1 1 1 0 1 0 1 0 0 1 0 0 0 1 0 0 1 0 0 1 0 0 1 0
 0 0 0 0 1 1 0 0 0 1 0 1 1 0 1 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 0 0 1
 0 0 0 1 0 0 0 1 0 1 0 1 0 0 0 1 0 1 1 1 0 0 1 1 0 1 1 1 1 1 1 0 0 0 0 0 1
 1 0 0 0 1 0 0 1 0 0 1 0 0 0 0 0 0 1 0 1 1 0 1 0 0 0 0 0 0 1 0 1]


In [94]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [95]:
bias = torch.rand(1, dtype=torch.float64, requires_grad= True)

X_train_tensor = X_train_tensor.to(torch.float32)
X_train_tensor = X_train_tensor.float()
y_train_tensor = y_train_tensor.float()

## Dataloaders and Dataset

In [96]:
from torch.utils.data import Dataset, DataLoader

In [97]:
class CustomDataloader(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return self.features.shape[0]
    
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [98]:
train_dataset = CustomDataloader(X_train_tensor, y_train_tensor)
test_dataset = CustomDataloader(y_train_tensor, y_test_tensor)

In [99]:
train_dataset[:]

(tensor([[-0.9954, -0.2402, -1.0230,  ..., -1.4423, -0.3388, -0.9180],
         [-0.2749,  0.2736, -0.3264,  ..., -1.1382, -0.3579, -0.5338],
         [-0.4863, -0.3883, -0.4311,  ..., -0.6488, -0.5662,  0.5363],
         ...,
         [-0.4975,  0.5653, -0.4994,  ...,  0.0725,  0.0953,  0.4819],
         [ 1.5136,  1.3059,  1.4368,  ...,  0.4748, -0.9844, -1.2757],
         [ 0.0282,  0.5468,  0.0602,  ...,  0.4455,  1.0796,  0.9693]]),
 tensor([0., 0., 0., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 0., 0.,
         0., 1., 0., 1., 0., 0., 0., 1., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
         0., 1., 1., 1., 1., 0., 1., 1., 0., 0., 1., 0., 0., 0., 0., 0., 1., 1.,
         0., 0., 1., 1., 0., 0., 1., 0., 0., 0., 0., 0., 1., 0., 0., 1., 0., 0.,
         1., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 1., 0., 1., 1., 1., 1.,
         0., 0., 1., 1., 0., 1., 1., 1., 0., 0., 1., 0., 1., 1., 0., 1., 1., 0.,
         1., 0., 0., 1., 0., 1., 1., 1., 0., 0., 1., 0., 1., 0., 0., 0.,

In [100]:
train_dataset[10]

(tensor([-1.4660, -0.8952, -1.4303, -1.1598, -0.9462, -0.5330, -0.5408, -0.6657,
          0.4344,  0.4671,  0.4959, -0.0222,  0.6859, -0.2061,  1.2172,  0.1686,
         -0.0175,  0.5051,  0.8876,  0.1255, -1.2924, -1.2787, -1.2425, -1.0117,
         -1.3565, -0.8393, -0.8871, -1.0508, -0.7602, -0.5327]),
 tensor(0.))

In [101]:
test_dataset[10]

(tensor(0.), tensor(0))

In [102]:
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle= True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle= True)

## Defining Model

In [103]:
class simpleNN(nn.Module):

    def __init__(self, num_features):
        super().__init__()
        self.linear = nn.Linear(num_features,1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, features):

        out = self.linear(features)
        out = self.sigmoid(out)

        return out

In [104]:
model = simpleNN(X_test_tensor.shape[1])   # number of input features/ Columns

learning_rate = 0.1
epochs = 45


optimizer = torch.optim.SGD(model.parameters(), lr= learning_rate)

loss_func = nn.BCELoss()

In [105]:
for name, param in model.named_parameters():
    print(name, param.shape)

linear.weight torch.Size([1, 30])
linear.bias torch.Size([1])


In [106]:
for epoch in range(epochs):

    for batch_features, batch_labels in train_dataloader:

        #FORWARD PASS
        y_pred = model(batch_features)

        #loss
        loss = loss_func(y_pred, batch_labels.view(-1,1))

        #  Zero grads
        optimizer.zero_grad()

        # Backward pass
        loss.backward()

        # Updating Gradients's parameters
        optimizer.step()


        print (f"{epoch} and loss {loss.item()}")

0 and loss 0.6386319398880005
0 and loss 0.4051891267299652
0 and loss 0.3925010561943054
0 and loss 0.3640853762626648
0 and loss 0.35975825786590576
0 and loss 0.2978096902370453
0 and loss 0.2962040305137634
0 and loss 0.2400316596031189
0 and loss 0.2252846509218216
0 and loss 0.27140188217163086
0 and loss 0.25249356031417847
0 and loss 0.23461462557315826
0 and loss 0.2296096384525299
0 and loss 0.3408716320991516
1 and loss 0.17886316776275635
1 and loss 0.2263154685497284
1 and loss 0.19548404216766357
1 and loss 0.15208826959133148
1 and loss 0.12911713123321533
1 and loss 0.1832331120967865
1 and loss 0.13477566838264465
1 and loss 0.20072545111179352
1 and loss 0.2125610113143921
1 and loss 0.1514725387096405
1 and loss 0.20523113012313843
1 and loss 0.12319581210613251
1 and loss 0.20347504317760468
1 and loss 0.28714650869369507
2 and loss 0.16258113086223602
2 and loss 0.1721927970647812
2 and loss 0.14703135192394257
2 and loss 0.11547460407018661
2 and loss 0.1730571836

In [107]:
# Model Evaluation

y_pred = y_pred.float()
with torch.no_grad():                     # Disable gradient calculation

    y_pred = model(X_test_tensor.float()) # Predict probabilities

    y_pred = (y_pred > 0.5).float()       # Convert probabilities to 0 or 1

    accuracy = (y_pred == y_test_tensor).float().mean()  # Calculate accuracy

    print(f"Accuracy: {accuracy.item()}") # Print accuracy

Accuracy: 0.551518440246582
